# N15 · MoE Router 与 Capacity

**关联 lab**: L05.5

**学习目标**: 把 "router 决定 token 去哪个 expert" 从概念变成可断言的 numpy 复现。手算 top-k softmax、aux loss、capacity overflow，并复现 router collapse 与 aux loss 修复的全过程。

**No-GPU 可完成度**: 100%。

**对应 MiniInfra**:
- `mini_infra/megatron/core/transformer/moe/router.py`
- `mini_infra/megatron/core/transformer/moe/capacity.py`
- `mini_infra/megatron/core/transformer/moe/alltoall.py`

**对应真实源码**: `github_repo/Megatron-LM/megatron/core/transformer/moe/`

## 1. Top-k softmax router

MoE 的 router 把每个 token 路由到 k 个 expert：

1. 对 token 算 logits ∈ ℝ^N (N = num_experts)
2. 取 top-k
3. softmax(top-k logits) → 权重
4. token 被复制 k 份送到对应 expert，输出加权求和

Switch (top-1) 简单但易 collapse；GShard / Mixtral (top-2) 是工业标准。

In [ ]:
import sys
from pathlib import Path
ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from mini_infra.megatron.core.transformer.moe.router import (
    route_tokens, tokens_per_expert, router_entropy,
    auxiliary_load_balance_loss, synthetic_logits, softmax
)

logits_normal = synthetic_logits(num_tokens=64, num_experts=8, collapse=False)
for top_k in [1, 2, 3]:
    routes = route_tokens(logits_normal, top_k=top_k)
    print(f'top-k={top_k}: 第一个 token 的路由 = {routes[0]}')

## 2. 健康路由 vs collapse

用 `synthetic_logits(collapse=True)` 模拟 expert 0 略占优的情况，看 routing 分布与 entropy 的差异：

In [ ]:
for collapse in [False, True]:
    logits = synthetic_logits(num_tokens=128, num_experts=8, collapse=collapse)
    routes = route_tokens(logits, top_k=2)
    counts = tokens_per_expert(routes, num_experts=8)
    entropy = router_entropy(routes, num_experts=8)
    aux = auxiliary_load_balance_loss(routes, num_experts=8)
    print(f'collapse={collapse}')
    print(f'  tokens_per_expert = {counts}')
    print(f'  p99/p50 ratio     = {max(counts) / sorted(counts)[len(counts)//2]:.2f}')
    print(f'  router_entropy    = {entropy:.4f}  (1.0=完美均匀, 0.0=全部去同一 expert)')
    print(f'  aux_loss          = {aux:.6f}  (越大越偏)')
    print()

**关键观察**: 
- 健康 router: entropy ≈ 0.95+，counts 接近均匀
- collapse: entropy 显著下降，某个 expert 拿走 30-50% token
- aux_loss 在 collapse 时显著高于健康——这就是用它当 loss 项的理由

**监控阈值**：
- entropy < 0.5 → 已 collapse，立即 alert
- p99 / p50 > 5 → 严重 imbalance

## 3. Aux loss 防 collapse 实验

模拟训练 100 步，每步 router 略偏向 expert 0 + aux_loss 拉它回来。看不同 aux_loss 系数的效果：

In [ ]:
import math

def simulate_training(aux_coef, steps=100, num_experts=8, num_tokens=64):
    """极简模拟：每步给 expert 0 +0.1 logit；aux loss 提供反向阻力"""
    expert_bias = [0.0] * num_experts
    history = []
    for step in range(steps):
        # 数据偏好让 expert 0 收到更多正反馈
        logits = synthetic_logits(num_tokens, num_experts, collapse=False)
        logits = [[v + expert_bias[i] for i, v in enumerate(row)] for row in logits]
        routes = route_tokens(logits, top_k=2)
        counts = tokens_per_expert(routes, num_experts)
        ent = router_entropy(routes, num_experts)
        aux = auxiliary_load_balance_loss(routes, num_experts)
        history.append({'step': step, 'entropy': ent, 'aux': aux, 'max_ratio': max(counts)/(sum(counts)/num_experts)})
        # "梯度"：偏向多数 expert，aux_loss 反向修正
        for i in range(num_experts):
            expert_bias[i] += 0.05 * (counts[i] / sum(counts) - 1.0/num_experts)
            expert_bias[i] -= aux_coef * 10 * (counts[i] / sum(counts) - 1.0/num_experts)
    return history

for coef in [0.0, 0.001, 0.01, 0.1]:
    h = simulate_training(coef, steps=200)
    final = h[-1]
    print(f'aux_coef={coef:>6}: final entropy={final["entropy"]:.4f}, '
          f'max_ratio={final["max_ratio"]:.2f}')

**结论**: aux_coef=0 时 entropy 单调下降到 collapse；aux_coef=0.01 把它稳住。这就是工业默认值。

**警告**：aux_coef 过大（如 1.0）会强制均匀，损害 router 学习能力。0.01-0.05 是甜区。

## 4. Capacity factor 与 token drop

每个 expert 最多接 `capacity = ceil(tokens × top_k / experts × cf)` 个 token。超出 → drop（或 reroute）。

扫 cf 看 overflow 率：

In [ ]:
from mini_infra.megatron.core.transformer.moe.capacity import apply_capacity, capacity_per_expert

num_tokens, num_experts, top_k = 256, 8, 2
for collapse in [False, True]:
    print(f'\n=== collapse={collapse} ===')
    for cf in [0.8, 1.0, 1.25, 1.5, 2.0]:
        cap = capacity_per_expert(num_tokens, num_experts, top_k, cf)
        logits = synthetic_logits(num_tokens, num_experts, collapse=collapse)
        routes = route_tokens(logits, top_k=top_k)
        result = apply_capacity(routes, num_experts, capacity_factor=cf)
        print(f'  cf={cf:>5}  capacity/expert={cap:>3}  '
              f'overflow={result["capacity_overflow_rate"]*100:>5.2f}%  '
              f'kept={result["kept"]}/{num_tokens*top_k}')

**观察**: 
- 健康 router 下 cf=1.25 已经 0% overflow
- collapse 时 cf=2.0 仍然 overflow——必须先解决 collapse，cf 不能掩盖根因

**cf 选择 rule**:
- 学术 / tiny: 1.25
- Mixtral 风格: 2.0
- DeepSeek-MoE 风格: 1.5

## 5. All-to-all 通信代价

MoE 一次 forward 需要两次 all-to-all（dispatch + combine）。EP 越大，通信总量越大。

In [ ]:
from mini_infra.megatron.core.transformer.moe.alltoall import (
    alltoall_cost_ms, dispatch_plan
)

logits = synthetic_logits(num_tokens=1024, num_experts=8, collapse=False)
routes = route_tokens(logits, top_k=2)

for ep_size in [1, 2, 4, 8]:
    plan = dispatch_plan(routes, num_experts=8, hidden_size=4096, ep_size=ep_size)
    print(f'EP={ep_size}  alltoall_ms={plan["alltoall_ms"]:.4f}  '
          f'tokens_per_expert={plan["tokens_per_expert"]}')

**注意**: 上面只算了一次 dispatch；MoE 实际是两次 alltoall（dispatch + combine），所以乘 2。

EP=8 不一定比 EP=4 快——alltoall payload 总量增加，但每 rank expert 数减半。是否回本看模型大小与 batch。

**rule of thumb**:
- 模型 < 1B: EP ≤ 4
- 模型 1B-7B: EP = 4-8
- 模型 70B+ MoE: EP = 8+

## 6. Active params vs total params

MoE 的卖点：active params 远小于 total params。8 个 expert + top-2 → active = 2/8 = 1/4 总参数量。

In [ ]:
from mini_infra.megatron.core.transformer.moe import moe_summary

for n_exp, k in [(4, 2), (8, 2), (16, 2), (8, 1)]:
    s = moe_summary(num_tokens=64, num_experts=n_exp, top_k=k, capacity_factor=1.25, ep_size=2)
    ratio = s['active_params'] / s['total_params']
    print(f'{n_exp:>3} experts × top-{k}: '
          f'active={s["active_params"]/1e6:>6.1f}M, '
          f'total={s["total_params"]/1e6:>6.1f}M, '
          f'active/total={ratio*100:.1f}%, '
          f'router_entropy={s["router_entropy"]:.3f}')

**关键经济性**: 
- 8×top-2: active 是 dense 同 hidden 的 1× 算力，但模型容量是 4×（更多知识）
- Mixtral 8×7B: 47B total, 13B active（计算成本接近 13B dense，能力接近 70B）
- DeepSeek-MoE 16B: 16.4B total, 2.8B active（极高 sparsity）

## 7. 自检问题

1. top-k=1（Switch）与 top-k=2（GShard/Mixtral）的训练稳定性差异？哪个更易 collapse？
2. aux_loss 系数从 0 提到 0.01，router_entropy 应该上升还是下降？
3. 8 expert + top-2 + cf=1.25 时，单 expert capacity 是多少（假设 256 token）？
4. EP=8 一次 forward 总通信字节数与 EP=4 比是几倍？
5. Mixtral 8×7B 的 active params 大约是 13B；如果改成 8 expert × top-3，active 大约多少？

## 8. 与 lab 对接

Lab 任务：在 Megatron 上跑 dense vs MoE(8, top-2)、EP=1/2/4/8 sweep、cf=1.0/1.25/2.0 对照、aux_loss 0/0.001/0.01 防 collapse。

Smoke：
```bash
torchrun --nproc_per_node=1 labs/l13_moe_ep/scripts/run_moe.py \
  --experts 4 --top-k 2 --hidden 128 --aux-loss 0.01
```

失败时按以下顺序查 ticket：
- entropy 持续下降 → `moe_router_collapse_001`
- overflow > 5% → `moe_capacity_overflow_002`
- alltoall 超时 / EP=8 比 EP=4 慢 → `moe_alltoall_slow_003`